# Task 9 — Hybrid Recommendation

Combines the Task 5 TF-IDF content model and Task 8 user-based collaborative filtering. Each model score is min-max normalized over its candidate list before a weighted blend. This is a ranking score, not a calibrated probability.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "dataset").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.recommender import TEDRecommender
model = TEDRecommender.from_csv(ROOT / "dataset/ted_talks_cleaned.csv", ROOT / "dataset/user_interactions.csv")
print(f"Loaded {len(model.talks):,} talks and {len(model.interactions):,} interactions across {len(model.user_item):,} users")

## Compare model outputs for a user

In [ ]:
user_id = str(model.user_item.index[0])
content = model.content_recommendations(user_id, top_n=10)
collaborative = model.collaborative_recommendations(user_id, top_n=10)
hybrid = model.hybrid_recommendations(user_id, top_n=10, content_weight=0.5)
comparison = pd.concat([content.assign(model="Content"), collaborative.assign(model="Collaborative"), hybrid.assign(model="Hybrid")])
display(comparison[["model", "talk_id", "title", "score"]].sort_values(["model", "score"], ascending=[True,False]))

## Try several users and weights

The output records separate rankings so overlaps and differences can be inspected. For content-only recommendations without an interaction profile, pass one or more `seed_talk_ids`.

In [ ]:
rows = []
for user in model.user_item.index[:5]:
    for method, result in [("content", model.content_recommendations(user, top_n=5)), ("collaborative", model.collaborative_recommendations(user, top_n=5)), ("hybrid", model.hybrid_recommendations(user, top_n=5, content_weight=0.5))]:
        for rank, rec in enumerate(result.itertuples(), 1): rows.append({"user_id":user,"model":method,"rank":rank,"talk_id":rec.talk_id,"title":rec.title,"score":rec.score})
user_comparison = pd.DataFrame(rows)
display(user_comparison)

## Observations

- Content recommendations use the user’s talks rated 4 or 5 as a TF-IDF profile; new users can supply selected seed talks.
- Collaborative recommendations use similarity between rating vectors and exclude already rated talks.
- Hybrid rankings remove duplicates by aggregating scores by talk ID, then sort descending. Sparse users can have few collaborative candidates; in that case the content side supplies candidates.
- Scores depend on the candidate set and should be interpreted as relative ranking scores.